In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
from tqdm import tqdm
import os

# ==========================================
# 0. Configuration & Hyperparameters
# ==========================================
CONFIG = {
    'LATENT_DIM': 640,          # Output of WRN-28-10
    'BATCH_SIZE': 64,           # Smaller batch size for stability with SAM
    'LR_MANIFOLD': 1e-3,        # Learning rate for Autoencoder
    'LR_CLASS': 0.05,           # Learning rate for Classifier (SGD)
    'LAMBDA_LOCAL': 1.0,        # Weight for topology loss
    'EPOCHS': 200,
    'PATIENCE': 20,
    'RHO': 0.05,                # SAM neighborhood size
    'CUTOUT_LEN': 16,           # Size of the black box
    'SEEN_CLASSES': [0, 1, 2, 3, 4],
    'UNSEEN_CLASSES': [5, 6, 7, 8, 9],
    'DEVICE': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'SAVE_PATH': './zsl_wrn_sam_best.pth'
}

print(f"Running on: {CONFIG['DEVICE']}")

# ==========================================
# 1. Advanced Data Transforms (Inpainting)
# ==========================================

class Cutout(object):
    """Randomly masks out a square patch from the image."""
    def __init__(self, n_holes, length):
        self.n_holes = n_holes
        self.length = length

    def __call__(self, img):
        h = img.size(1)
        w = img.size(2)
        mask = np.ones((h, w), np.float32)
        for n in range(self.n_holes):
            y = np.random.randint(h)
            x = np.random.randint(w)
            y1 = np.clip(y - self.length // 2, 0, h)
            y2 = np.clip(y + self.length // 2, 0, h)
            x1 = np.clip(x - self.length // 2, 0, w)
            x2 = np.clip(x + self.length // 2, 0, w)
            mask[y1: y2, x1: x2] = 0.
        mask = torch.from_numpy(mask)
        mask = mask.expand_as(img)
        return img * mask

class TwoCropTransform:
    """Returns (Corrupted, Clean) for Inpainting tasks."""
    def __init__(self, clean_transform, corrupt_transform):
        self.clean_transform = clean_transform
        self.corrupt_transform = corrupt_transform

    def __call__(self, x):
        clean = self.clean_transform(x)
        corrupt = self.corrupt_transform(x)
        return corrupt, clean

# Base normalization for CIFAR-10
NORMALIZE = transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

# 1. Clean Transform (Target)
transform_clean = transforms.Compose([
    transforms.ToTensor(),
    NORMALIZE
])

# 2. Corrupt Transform (Input) - Includes Cutout
transform_corrupt = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    NORMALIZE,
    Cutout(n_holes=1, length=CONFIG['CUTOUT_LEN'])
])

# ==========================================
# 2. Optimizer: SAM (Sharpness-Aware Minimization)
# ==========================================
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"] if p.grad is not None
            ]), p=2
        )
        return norm

    def zero_grad(self, set_to_none=False):
        self.base_optimizer.zero_grad(set_to_none)

# ==========================================
# 3. Model Architecture (WideResNet-28-10)
# ==========================================

class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
        super(BasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.convShortcut = (not self.equalInOut) and nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False) or None

    def forward(self, x):
        if not self.equalInOut: x = self.relu1(self.bn1(x))
        else: out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0: out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.equalInOut else self.convShortcut(x), out)

class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super(NetworkBlock, self).__init__()
        self.layer = self._make_layer(block, in_planes, out_planes, nb_layers, stride, dropRate)
    def _make_layer(self, block, in_planes, out_planes, nb_layers, stride, dropRate):
        layers = []
        for i in range(int(nb_layers)):
            layers.append(block(i == 0 and in_planes or out_planes, out_planes, i == 0 and stride or 1, dropRate))
        return nn.Sequential(*layers)
    def forward(self, x): return self.layer(x)

class WRNEncoder(nn.Module):
    def __init__(self, depth=28, widen_factor=10, dropRate=0.0):
        super(WRNEncoder, self).__init__()
        nChannels = [16, 16*widen_factor, 32*widen_factor, 64*widen_factor]
        n = (depth - 4) / 6
        block = BasicBlock
        self.conv1 = nn.Conv2d(3, nChannels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, nChannels[0], nChannels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, nChannels[1], nChannels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, nChannels[2], nChannels[3], block, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(nChannels[3])
        self.relu = nn.ReLU(inplace=True)
        self.nChannels = nChannels[3]

        for m in self.modules():
            if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): m.weight.data.fill_(1); m.bias.data.zero_()

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.avg_pool2d(out, 8)
        flat = out.view(-1, self.nChannels)
        
        # IMPROVEMENT 2: Hypersphere Constraint (L2 Normalize)
        # This makes cosine similarity == euclidean distance
        return F.normalize(flat, p=2, dim=1)

class SimpleDecoder(nn.Module):
    def __init__(self, latent_dim=CONFIG['LATENT_DIM']):
        super(SimpleDecoder, self).__init__()
        self.linear = nn.Linear(latent_dim, 256*4*4)
        self.deconv1 = nn.ConvTranspose2d(256, 128, 4, 2, 1) # 4 -> 8
        self.deconv2 = nn.ConvTranspose2d(128, 64, 4, 2, 1)  # 8 -> 16
        self.deconv3 = nn.ConvTranspose2d(64, 32, 4, 2, 1)   # 16 -> 32
        self.final = nn.Conv2d(32, 3, 3, 1, 1)
        self.activation = nn.LeakyReLU(0.2)
        
    def forward(self, z):
        x = self.linear(z)
        x = x.view(-1, 256, 4, 4)
        x = self.activation(self.deconv1(x))
        x = self.activation(self.deconv2(x))
        x = self.activation(self.deconv3(x))
        x = torch.sigmoid(self.final(x)) # Output 0-1 range
        return x

class ProjectionHead(nn.Module):
    """Decouples the Manifold Representation from the Classification Logic."""
    def __init__(self, input_dim=CONFIG['LATENT_DIM'], hidden_dim=256, output_dim=len(CONFIG['SEEN_CLASSES'])):
        super(ProjectionHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

# ==========================================
# 4. Losses & Utilities
# ==========================================

def compute_topology_loss(imgs_clean, z, k=5):
    """
    Enforces that neighbors in pixel space (clean images) 
    remain neighbors in latent space.
    """
    batch_size = imgs_clean.size(0)
    if batch_size <= k: return torch.tensor(0.0).to(CONFIG['DEVICE'])

    # Flatten images for distance calculation
    imgs_flat = imgs_clean.view(batch_size, -1)
    
    # Calculate distances in Pixel Space (Ground Truth)
    dist_input = torch.cdist(imgs_flat, imgs_flat, p=2)
    _, indices = dist_input.topk(k + 1, largest=False)
    neighbor_indices = indices[:, 1:] # Exclude self
    
    # Get corresponding Latent Vectors
    z_neighbors = z[neighbor_indices] # Shape: (B, k, latent_dim)
    z_expanded = z.unsqueeze(1)       # Shape: (B, 1, latent_dim)
    
    # MSE Loss between point and its neighbors in latent space
    loss = F.mse_loss(z_expanded.expand_as(z_neighbors), z_neighbors)
    return loss

# ==========================================
# 5. Training Engine
# ==========================================

def train_framework():
    # --- A. Data Preparation ---
    print("==> Preparing Data...")
    
    # Dual transform wrapper
    dual_transform = TwoCropTransform(transform_clean, transform_corrupt)
    
    # Note: We load dataset twice to handle different transforms cleanly if needed, 
    # but here we apply the DualTransform to the TRAINING set only.
    full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=dual_transform)
    test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_clean)

    # Indices
    targets = torch.tensor(full_dataset.targets)
    seen_mask = torch.isin(targets, torch.tensor(CONFIG['SEEN_CLASSES']))
    unseen_mask = torch.isin(targets, torch.tensor(CONFIG['UNSEEN_CLASSES']))
    
    seen_indices = torch.where(seen_mask)[0].tolist()
    unseen_indices = torch.where(unseen_mask)[0].tolist()
    
    # Validation Split (10% of Seen Data)
    np.random.seed(42)
    np.random.shuffle(seen_indices)
    val_split = len(seen_indices) // 10
    val_indices = seen_indices[:val_split]
    train_seen_indices = seen_indices[val_split:]
    
    print(f"Data Stats: {len(train_seen_indices)} Seen Train | {len(val_indices)} Seen Val | {len(unseen_indices)} Unseen (Transductive)")

    # Loaders
    # 1. Manifold Loader: Uses ALL data (Seen + Unseen) for unsupervised learning
    loader_manifold = DataLoader(full_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=2, drop_last=True)
    
    # 2. Classifier Loader: Uses ONLY SEEN data
    loader_classifier = DataLoader(Subset(full_dataset, train_seen_indices), batch_size=CONFIG['BATCH_SIZE'], shuffle=True, num_workers=2, drop_last=True)
    
    # 3. Validation Loader
    loader_val = DataLoader(Subset(full_dataset, val_indices), batch_size=100, shuffle=False, num_workers=2)

    # --- B. Model Init ---
    encoder = WRNEncoder().to(CONFIG['DEVICE'])
    decoder = SimpleDecoder().to(CONFIG['DEVICE'])
    proj_head = ProjectionHead().to(CONFIG['DEVICE'])
    
    # --- C. Optimizers ---
    # Phase 1: Adam for fast reconstruction convergence
    opt_manifold = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=CONFIG['LR_MANIFOLD'])
    
    # Phase 2: SAM for robust classification
    base_optimizer = torch.optim.SGD
    opt_class = SAM(list(encoder.parameters()) + list(proj_head.parameters()), 
                    base_optimizer, lr=CONFIG['LR_CLASS'], momentum=0.9, weight_decay=5e-4, rho=CONFIG['RHO'])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_class.base_optimizer, T_max=CONFIG['EPOCHS'])

    # --- D. Training Loop ---
    best_val_acc = 0.0
    patience_counter = 0
    
    print("\n" + "="*50)
    print("STARTING SOTA TRAINING (WRN + SAM + Inpainting)")
    print("="*50)

    for epoch in range(CONFIG['EPOCHS']):
        
        # ---------------------------------------
        # Phase 1: Manifold Learning (Unsupervised)
        # Task: Inpainting + Topology Preservation
        # ---------------------------------------
        encoder.train()
        decoder.train()
        total_recon = 0
        total_topo = 0
        
        # Iterate over ALL data (Transductive)
        for (corrupt, clean), _ in tqdm(loader_manifold, desc=f"Ep {epoch+1} [Manifold]", leave=False):
            corrupt, clean = corrupt.to(CONFIG['DEVICE']), clean.to(CONFIG['DEVICE'])
            
            opt_manifold.zero_grad()
            
            z = encoder(corrupt)   # Encode damaged image
            recon = decoder(z)     # Try to fix it
            
            # Loss 1: Reconstruction against CLEAN target (Inpainting)
            loss_recon = F.mse_loss(recon, clean)
            
            # Loss 2: Topology against CLEAN inputs (Structure)
            loss_topo = compute_topology_loss(clean, z)
            
            loss = loss_recon + (CONFIG['LAMBDA_LOCAL'] * loss_topo)
            loss.backward()
            opt_manifold.step()
            
            total_recon += loss_recon.item()
            total_topo += loss_topo.item()

        # ---------------------------------------
        # Phase 2: Classification (Supervised)
        # Task: Classify Seen Classes with SAM
        # ---------------------------------------
        proj_head.train()
        total_cls = 0
        
        for (corrupt, _), labels in tqdm(loader_classifier, desc=f"Ep {epoch+1} [Classify]", leave=False):
            corrupt, labels = corrupt.to(CONFIG['DEVICE']), labels.to(CONFIG['DEVICE'])
            
            # Re-map labels (0-5) are fine. 
            # Note: CIFAR labels are already 0-9. Seen are 0-5.
            
            # SAM Step 1
            z = encoder(corrupt)
            preds = proj_head(z)
            loss = F.cross_entropy(preds, labels)
            loss.backward()
            opt_class.first_step(zero_grad=True)
            
            # SAM Step 2
            z_sharp = encoder(corrupt)
            preds_sharp = proj_head(z_sharp)
            F.cross_entropy(preds_sharp, labels).backward()
            opt_class.second_step(zero_grad=True)
            
            total_cls += loss.item()
            
        scheduler.step()

        # ---------------------------------------
        # Validation
        # ---------------------------------------
        encoder.eval()
        proj_head.eval()
        correct = 0; total = 0
        
        with torch.no_grad():
            for (_, clean), labels in loader_val:
                clean, labels = clean.to(CONFIG['DEVICE']), labels.to(CONFIG['DEVICE'])
                z = encoder(clean) # Validation uses clean images
                preds = proj_head(z)
                predicted = preds.argmax(1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = 100. * correct / total
        
        # Logging
        avg_recon = total_recon / len(loader_manifold)
        avg_cls = total_cls / len(loader_classifier)
        
        print(f"Epoch {epoch+1:02d} | Recon: {avg_recon:.4f} | Cls: {avg_cls:.4f} | Val Acc: {val_acc:.2f}%", end="")
        
        # Save Best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save({
                'encoder': encoder.state_dict(),
                'proj_head': proj_head.state_dict(),
                'decoder': decoder.state_dict(),
                'config': CONFIG
            }, CONFIG['SAVE_PATH'])
            print(" (*)")
        else:
            patience_counter += 1
            print(f" ({patience_counter})")
            
        if patience_counter >= CONFIG['PATIENCE']:
            print("Early Stopping Triggered.")
            break

    return encoder, full_dataset

# ==========================================
# 6. Zero-Shot Inference
# ==========================================

def evaluate_zero_shot(encoder, dataset):
    """
    Performs Zero-Shot Classification using Nearest Prototype.
    prototypes are calculated using the 'clean' images.
    """
    print("\n" + "="*50)
    print("ZERO-SHOT EVALUATION")
    print("="*50)
    
    encoder.eval()
    
    # 1. Compute Prototypes for ALL classes (using labels to cluster)
    # In a real ZSL setting, you might use semantic vectors (word2vec), 
    # but in Transductive ZSL, we typically cluster the unlabeled data.
    # Here, we check the quality of the 'Oracle' prototypes (Best Case Scenario).
    
    centroids = {}
    counts = {}
    
    # Helper loader for clean inference
    # We must strip the DualTransform and just get clean images
    clean_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform_clean)
    loader = DataLoader(clean_dataset, batch_size=100, shuffle=False, num_workers=2)
    
    print("Computing Class Centroids...")
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Centroids"):
            imgs = imgs.to(CONFIG['DEVICE'])
            z_batch = encoder(imgs)
            
            for i in range(len(labels)):
                lbl = labels[i].item()
                if lbl not in centroids:
                    centroids[lbl] = z_batch[i]
                    counts[lbl] = 1
                else:
                    centroids[lbl] += z_batch[i]
                    counts[lbl] += 1
    
    # Average them
    for k in centroids:
        centroids[k] /= counts[k]
        
    # 2. Test on UNSEEN classes
    print(f"Testing on Unseen Classes: {CONFIG['UNSEEN_CLASSES']}")
    
    # Filter dataset
    targets = torch.tensor(clean_dataset.targets)
    unseen_idx = torch.where(torch.isin(targets, torch.tensor(CONFIG['UNSEEN_CLASSES'])))[0]
    unseen_subset = Subset(clean_dataset, unseen_idx)
    loader_test = DataLoader(unseen_subset, batch_size=100, shuffle=False)
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader_test, desc="Inference"):
            imgs, labels = imgs.to(CONFIG['DEVICE']), labels.to(CONFIG['DEVICE'])
            z_batch = encoder(imgs)
            
            for i, label in enumerate(labels):
                z = z_batch[i]
                
                # Find nearest centroid among UNSEEN classes
                min_dist = float('inf')
                pred = -1
                
                for c_lbl in CONFIG['UNSEEN_CLASSES']:
                    # Euclidian distance (works well with L2 normalized vectors)
                    dist = torch.dist(z, centroids[c_lbl])
                    if dist < min_dist:
                        min_dist = dist
                        pred = c_lbl
                
                if pred == label.item():
                    correct += 1
                total += 1
                
    acc = 100. * correct / total
    print(f"\nFinal Zero-Shot Accuracy: {acc:.2f}%")
    print(f"Model saved to: {CONFIG['SAVE_PATH']}")

if __name__ == "__main__":
    trained_encoder, full_ds = train_framework()
    
    # Load best weights for evaluation
    checkpoint = torch.load(CONFIG['SAVE_PATH'])
    trained_encoder.load_state_dict(checkpoint['encoder'])
    
    evaluate_zero_shot(trained_encoder, full_ds)

Running on: cuda
==> Preparing Data...
Files already downloaded and verified
Files already downloaded and verified
Data Stats: 22500 Seen Train | 2500 Seen Val | 25000 Unseen (Transductive)

STARTING SOTA TRAINING (WRN + SAM + Inpainting)


Epoch 01 | Recon: 1.2460 | Cls: 1.3195 | Val Acc: 51.28% (*)


Epoch 02 | Recon: 1.1669 | Cls: 1.1478 | Val Acc: 57.28% (*)


Epoch 03 | Recon: 1.1367 | Cls: 1.0850 | Val Acc: 59.48% (*)


Epoch 04 | Recon: 1.1241 | Cls: 1.0229 | Val Acc: 60.28% (*)


...

Computing Class Centroids...


Centroids: 100%|██████████| 500/500 [00:11<00:00, 42.01it/s]


Testing on Unseen Classes: [5, 6, 7, 8, 9]


Inference: 100%|██████████| 250/250 [00:09<00:00, 25.57it/s]


Final Zero-Shot Accuracy: 78.76%
Model saved to: ./zsl_wrn_sam_best.pth
